# NUTDTS 816 Time Series Analysis
## L10 Regression with ARIMA errors and Fourier terms

Lab notebook for Chapter 5 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Carried forward from Lab 9 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import tsdata
ap = tsdata.airpassengers(); y = np.log(ap)
train, test = y[:'1958-12'], y['1959-01':]     # hold out the last two years
d = train.diff().diff(12).dropna()
fig, axes = plt.subplots(1, 3, figsize=(11, 3), gridspec_kw={'width_ratios': [2, 1, 1]})
d.plot(ax=axes[0], lw=0.9, title='(1−B)(1−B¹²) log passengers, training set'); axes[0].set_xlabel('')
plot_acf(d, lags=36, ax=axes[1], title='ACF'); plot_pacf(d, lags=36, ax=axes[2], title='PACF')
for ax in axes[1:]: ax.set_ylim(-0.6, 1)
_caption = 'After one ordinary and one seasonal difference: a single negative spike at lag 1 and another at lag 12 in the ACF, with the PACF decaying at the seasonal lags. The signature of MA(1) × seasonal MA(1).'

In [ ]:
cands = [((0,1,1),(0,1,1,12)), ((1,1,0),(1,1,0,12)), ((0,1,1),(1,1,0,12)), ((1,1,0),(0,1,1,12)), ((0,1,2),(0,1,1,12)), ((1,1,1),(0,1,1,12))]
rows = []
for o, so in cands:
    f = ARIMA(train, order=o, seasonal_order=so).fit()
    lb = acorr_ljungbox(f.resid[13:], lags=[24], model_df=sum(o[::2]) + sum(so[:3:2]), return_df=True)
    rows.append({'order': f'{o}{so[:3]}12', 'AICc': round(f.aicc, 1), 'BIC': round(f.bic, 1), 'LB(24) p': round(lb.lb_pvalue.iloc[0], 3)})
print(pd.DataFrame(rows).sort_values('AICc').to_string(index=False))

In [ ]:
air = ARIMA(train, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit()
print(air.summary().tables[1])
fig = air.plot_diagnostics(figsize=(10, 5.5))
_caption = 'Airline model diagnostics on the training set: residuals are close to white noise and roughly normal.'

In [ ]:
h = len(test)
fc = air.get_forecast(h); ci = fc.conf_int(alpha=0.2)
snaive = pd.Series(train[-12:].values.tolist() * 2, index=test.index)     # seasonal naive in logs
ax = np.exp(y['1955':]).plot(figsize=(9, 3.4), lw=1, label='observed (incl. 1959-60 hold-out)')
np.exp(fc.predicted_mean).plot(ax=ax, color='#B8860B', lw=2, label='airline model')
ax.fill_between(ci.index, np.exp(ci.iloc[:, 0]), np.exp(ci.iloc[:, 1]), color='#B8860B', alpha=0.2, label='80% PI')
np.exp(snaive).plot(ax=ax, color='#555555', lw=1.2, ls='--', label='seasonal naive')
ax.axvline(test.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('Two-year forecast of airline passengers from December 1958')
act = np.exp(test)
for name, f_ in [('Airline model', np.exp(fc.predicted_mean)), ('Seasonal naive', np.exp(snaive))]:
    print(f'{name:16s} MAE = {(f_ - act).abs().mean():6.1f}   RMSE = {np.sqrt(((f_ - act)**2).mean()):6.1f}   (thousand passengers)')
_caption = 'The airline model tracks both the trend and the seasonal pattern over the two held-out years; the seasonal naive repeats 1958 and falls behind the growth.'

In [ ]:
from statsmodels.tsa.seasonal import STL
grid = tsdata.nigeria_grid(); gtr, gte = grid[:'2025-06'], grid['2025-07':]; h = len(gte)

# (a) SARIMA by automatic selection (pmdarima), then (b) STL + ARIMA, (c) seasonal naive
import pmdarima as pm
auto = pm.auto_arima(gtr, seasonal=True, m=12, stepwise=True, information_criterion='aicc', suppress_warnings=True, D=1)
print('auto_arima chose:', auto.order, auto.seasonal_order)
sar = ARIMA(gtr, order=auto.order, seasonal_order=auto.seasonal_order).fit()
f_sar = sar.get_forecast(h).predicted_mean

stl = STL(gtr, period=12, seasonal=13, robust=True).fit()
sa = gtr - stl.seasonal
auto_sa = pm.auto_arima(sa, seasonal=False, stepwise=True, information_criterion='aicc', suppress_warnings=True)
print('ARIMA on seasonally adjusted series:', auto_sa.order)
f_sa = ARIMA(sa, order=auto_sa.order, trend='t' if auto_sa.order[1] == 1 else 'c').fit().get_forecast(h).predicted_mean
seas_fwd = pd.Series(np.tile(stl.seasonal[-12:].values, 2)[:h], index=gte.index)    # last year's seasonal component carried forward
f_stl = f_sa + seas_fwd
f_sn = pd.Series(np.tile(gtr[-12:].values, 2)[:h], index=gte.index)

ax = grid['2022':].plot(figsize=(9, 3.4), lw=1, label='observed')
f_sar.plot(ax=ax, lw=2, color='#B8860B', label=f'SARIMA{auto.order}{auto.seasonal_order[:3]}12')
f_stl.plot(ax=ax, lw=1.5, color='#2F6DB5', ls='--', label='STL + ARIMA'); f_sn.plot(ax=ax, lw=1, color='#555555', ls=':', label='seasonal naive')
ax.axvline(gte.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('Grid generation (simulated): 12-month forecasts from June 2025')
for name, f_ in [('SARIMA', f_sar), ('STL + ARIMA', f_stl), ('Seasonal naive', f_sn)]:
    print(f'{name:16s} MAE = {(f_ - gte).abs().mean():6.0f} MW   RMSE = {np.sqrt(((f_ - gte)**2).mean()):6.0f} MW')
_caption = 'On a noisy series with a modest trend the ranking changes: STL + ARIMA is best, the seasonal naive is second, and the automatically selected SARIMA, whose seasonal AR(2) chases the last two years\' pattern, is worst. The collapses in the hold-out year are unforecastable by any of them.'

## Regression with ARIMA errors and Fourier terms

### 5.6 Regression with time series data: what goes wrong

In [ ]:
rng = np.random.default_rng(1)
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
x = np.cumsum(rng.normal(size=200)); yv = np.cumsum(rng.normal(size=200))   # two independent random walks
ols = sm.OLS(yv, sm.add_constant(x)).fit()
print(f'Regressing one independent random walk on another: slope t-stat = {ols.tvalues[1]:.2f}, p = {ols.pvalues[1]:.4f}, R² = {ols.rsquared:.3f}, Durbin-Watson = {durbin_watson(ols.resid):.3f}')
ols_d = sm.OLS(np.diff(yv), sm.add_constant(np.diff(x))).fit()
print(f'Same regression in first differences:               slope t-stat = {ols_d.tvalues[1]:.2f}, p = {ols_d.pvalues[1]:.4f}, R² = {ols_d.rsquared:.3f}, Durbin-Watson = {durbin_watson(ols_d.resid):.3f}')

### 5.8 Worked example III: inflation on oil and exchange-rate changes with an intervention (simulated)

In [ ]:
cpi = tsdata.nigeria_cpi(); fx = tsdata.nigeria_fx(); oilp = tsdata.bonny_light()
df = pd.DataFrame({'infl': 100 * np.log(cpi).diff(), 'd_oil_l1': 100 * np.log(oilp).diff().shift(1), 'd_fx_l1': 100 * np.log(fx).diff().shift(1)}).dropna()
df['regime'] = (df.index >= '2023-06-01').astype(float)
X = df[['d_oil_l1', 'd_fx_l1', 'regime']]
ols = sm.OLS(df['infl'], sm.add_constant(X)).fit()
print('OLS (standard errors assume independent errors):'); print(ols.summary().tables[1])
print(f'Durbin-Watson = {durbin_watson(ols.resid):.3f}  (2 = no first-order autocorrelation)')
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
plot_acf(ols.resid, lags=24, ax=axes[0], title='OLS residual ACF'); plot_pacf(ols.resid, lags=24, ax=axes[1], title='PACF')
for ax in axes: ax.set_ylim(-0.5, 1)
_caption = 'The OLS residuals are autocorrelated (a Durbin-Watson statistic of about 1.65 and a significant lag-1 autocorrelation): the standard errors above cannot be trusted, and an ARMA error model is indicated.'

In [ ]:
rows = []
for o in [(1, 0, 0), (2, 0, 0), (1, 0, 1), (0, 0, 1)]:
    f = ARIMA(df['infl'], exog=X, order=o).fit(); rows.append((o, round(f.aicc, 1)))
print('AICc by error model:', rows)
reg = ARIMA(df['infl'], exog=X, order=(1, 0, 1)).fit()
print(reg.summary().tables[1])
lb = acorr_ljungbox(reg.resid, lags=[12], model_df=2, return_df=True); print(f'Ljung-Box(12) on innovations: p = {lb.lb_pvalue.iloc[0]:.3f}')

In [ ]:
h = 12; future_idx = pd.date_range(df.index[-1], periods=h + 1, freq='MS')[1:]
Xf = pd.DataFrame({'d_oil_l1': [df['d_oil_l1'][-1]] + [df['d_oil_l1'][-12:].mean()] * (h - 1),
                   'd_fx_l1': [df['d_fx_l1'][-1]] + [df['d_fx_l1'][-12:].mean()] * (h - 1), 'regime': 1.0}, index=future_idx)
fc = reg.get_forecast(h, exog=Xf); ci = fc.conf_int(alpha=0.2)
ax = df['infl']['2022':].plot(figsize=(9, 3.2), lw=1, label='monthly inflation (%)')
fc.predicted_mean.plot(ax=ax, lw=2, color='#B8860B', label='dynamic regression forecast (scenario: recent average oil and FX changes)')
ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color='#B8860B', alpha=0.2); ax.legend(fontsize=8); ax.set_xlabel('')
print('Implied 12-month cumulative inflation under the scenario: {:.1f}%'.format(100 * (np.exp(fc.predicted_mean.sum() / 100) - 1)))
_caption = 'The forecast converges quickly to the regime mean plus the scenario contribution of the regressors; the interval reflects only the innovation variance, not the uncertainty in the scenario itself.'

### 5.9 Worked example IV: Fourier terms for daily data with two seasonal periods

In [ ]:
dem = tsdata.daily_demand()       # simulated daily demand (GWh) with weekly and annual seasonality, 2022-2025
def fourier(idx, period, K):
    t = np.arange(len(idx)); cols = {}
    for k in range(1, K + 1):
        cols[f's{period}_{k}'] = np.sin(2 * np.pi * k * t / period); cols[f'c{period}_{k}'] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=idx)
full_idx = pd.date_range(dem.index[0], periods=len(dem) + 28, freq='D')
F = pd.concat([fourier(full_idx, 7, 3), fourier(full_idx, 365.25, 4)], axis=1)
dtr, dte = dem[:'2025-11-02'], dem['2025-11-03':'2025-11-30']      # hold out four holiday-free weeks in November
har = ARIMA(dtr, exog=F.loc[dtr.index], order=(2, 0, 1)).fit()
fc = har.get_forecast(len(dte), exog=F.loc[dte.index]); ci = fc.conf_int(alpha=0.2)
ax = dem['2025-09-01':'2025-11-30'].plot(figsize=(9, 3.2), lw=1, label='observed'); fc.predicted_mean.plot(ax=ax, lw=2, color='#B8860B', label='Fourier (K=3 weekly, K=4 annual) + ARMA(2,1) errors')
ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color='#B8860B', alpha=0.2); ax.legend(fontsize=8); ax.set_xlabel('')
snaive7 = pd.Series(np.tile(dtr[-7:].values, 4), index=dte.index)
print(f'MAE over 4 weeks: harmonic regression = {(fc.predicted_mean - dte).abs().mean():.2f} GWh;  weekly seasonal naive = {(snaive7 - dte).abs().mean():.2f} GWh')
lb = acorr_ljungbox(har.resid, lags=[14], model_df=3, return_df=True); print(f'Ljung-Box(14) p = {lb.lb_pvalue.iloc[0]:.3f}')
_caption = 'Dynamic harmonic regression captures the weekend dips and the slow annual swing in a single model and beats the weekly seasonal naive by a wide margin over a holiday-free month. Public holidays (for example the dips on 1 October and 25 December) need their own dummies; that is Exercise 6.'

## Exercises

5. For the `uschange` data, fit a regression of consumption growth on income growth with (a) OLS and (b) ARMA errors chosen by AICc. Compare the coefficient on income and its standard error. Forecast consumption for eight quarters under the scenario that income growth equals its historical mean.
6. Using the `daily_demand()` series, add a public-holiday dummy to the harmonic regression of Section 5.9 and report its coefficient and its effect on the four-week MAE.
7. Explain in one paragraph to a non-technical manager why "we regressed inflation on the oil price and got $R^2 = 0.9$" is not, by itself, evidence that oil drives inflation.

In [ ]:
# Your work here
